<a href="https://colab.research.google.com/github/raihanewubd/CSE475/blob/main/exp3_ensembles_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE475 — Experiment 3: Supervised Learning II — Ensembles, Boosting and Model Evaluation

**Related theory:** Module 2 – Ensemble Learning and Performance Evaluation · **Scheduled:** Week 3 (2-hour laboratory session)

### Objective
To study how bagging, boosting and meta-ensembles improve on a single learner, and to evaluate every model with cross-validation and the full metric set rather than a single accuracy figure.

### How this notebook is organised
One section per task in the lab sheet (Tasks 1–9), then the three analysis questions. Every result table and figure has a caption cell directly beneath it. The notebook runs top to bottom with `SEED = 42`; if you change the seed, say so in the report.

### Rules that apply throughout
1. **The test set is touched once**, in Task 8, after the model and the decision threshold are frozen. Every comparison before that uses stratified 5-fold cross-validation on the training set only.
2. **Report variance, not one number.** Every CV score is reported as mean ± standard deviation across folds.
3. **Preprocessing lives inside the pipeline**, so it is fitted inside each fold and never sees the validation fold. Fitting a scaler on the whole training set before CV is a (mild) leak.

> **Dataset note.** The lab sheet asks for the Experiment 2 dataset for Tasks 1–5 and a moderately imbalanced dataset for the evaluation tasks. This notebook ships with the IBM Telco Customer Churn dataset (7,043 customers, 26.5% churn) as the default for *both* parts so it runs end to end. To use your Experiment 2 dataset, replace the loader in §0.2 and re-run Tasks 1–5; the cells are dataset-agnostic as long as you return `X` (DataFrame) and `y` (0/1 integer Series).

## 0. Setup
### 0.1 Imports and seed

In [ ]:
import time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, StratifiedKFold,
    cross_validate, cross_val_predict, GridSearchCV)
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
    f1_score, accuracy_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, auc, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.precision", 4)
print("numpy", np.__version__, "| pandas", pd.__version__)
import sklearn, xgboost; print("sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)

### 0.2 Load the data

The loader must return `X` (a DataFrame of raw features) and `y` (integer 0/1). **Document the class mapping** — here `Churn == "Yes"` → 1 (positive class = the event we want to catch), `"No"` → 0.

The Telco file has one trap worth knowing: `TotalCharges` is stored as text and contains 11 blank strings for customers with zero tenure. We coerce it to numeric and let the imputer in the pipeline handle the resulting NaNs — we do **not** drop rows or fill values here, because that would be a decision taken outside the CV loop.

In [ ]:
URL = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")

def load_telco(path=URL):
    df = pd.read_csv(path)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    y = (df["Churn"] == "Yes").astype(int)          # mapping: Yes -> 1, No -> 0
    X = df.drop(columns=["customerID", "Churn"])
    return X, y

# ---- To use your Experiment 2 dataset, replace the line below ----
X, y = load_telco()

print("Shape (rows, feature columns):", X.shape)
print("First 10 rows:")
display(pd.concat([X, y.rename("Churn (target)")], axis=1).head(10))

#### Column-by-column EDA

One row per column: storage type, missing count, number of distinct values, and either the value counts (categorical) or the five-number summary (numeric). Read the `distinct` column first: a numeric-looking column with 2 distinct values (`SeniorCitizen`) is really a binary flag, and a text column with 3 distinct values that includes "No internet service" is encoding a dependency on another column.

In [ ]:
rows = []
for c in X.columns:
    col = X[c]
    is_num = pd.api.types.is_numeric_dtype(col)
    summary = (f"min {col.min():g} | median {col.median():g} | mean {col.mean():.2f} | max {col.max():g}"
               if is_num else
               " | ".join(f"{k}: {v}" for k, v in col.value_counts().items()))
    rows.append(dict(column=c, dtype=str(col.dtype), kind="numeric" if is_num else "categorical",
                     missing=int(col.isna().sum()), distinct=col.nunique(), **{"values / summary": summary}))
eda = pd.DataFrame(rows).set_index("column")
with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(eda)

**Table 0a.** Column summary. Only `TotalCharges` has missing values (11, all zero-tenure customers). `SeniorCitizen` is numeric in storage but binary in meaning; it is left in the numeric branch, where standardising a 0/1 column is harmless.

In [ ]:
print("Numeric columns:")
display(X.describe().T.round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, c in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    for k, lab, col in [(0, "No churn", "#2E5C8A"), (1, "Churn", "#BD582C")]:
        ax.hist(X.loc[y == k, c].dropna(), bins=30, alpha=0.6, density=True, label=lab, color=col)
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

**Figure 0a.** Distributions of the three continuous columns, split by class (density-normalised so the two classes are comparable despite the 73/27 imbalance). Churners cluster at short tenure and high monthly charges; `TotalCharges` is essentially tenure × monthly charges, so expect the three to share importance in Task 9.

In [ ]:
# Churn rate within each level of every categorical column: which levels carry signal?
fig, axes = plt.subplots(3, 5, figsize=(16, 9)); axes = axes.ravel()
for ax, c in zip(axes, cat_cols_all := X.select_dtypes(exclude=np.number).columns):
    rate = y.groupby(X[c]).mean().sort_values()
    rate.plot.barh(ax=ax, color="#3C7A5A"); ax.axvline(y.mean(), ls="--", color="grey")
    ax.set_title(c, fontsize=10); ax.set_xlim(0, 0.5); ax.set_ylabel("")
plt.tight_layout(); plt.show()

**Figure 0b.** Churn rate per level of each categorical column; the dashed line is the overall rate (0.265). Columns whose bars all sit on the line (`gender`, `PhoneService`) carry almost no signal on their own; `Contract`, `InternetService` and `PaymentMethod` separate strongly. Keep this figure in mind when reading the Task-9 importances.

### 0.3 One stratified train/test split, then put the test set away

80/20, stratified on `y`. Stratification is what makes the two shares below identical to four decimals; a plain random split would give the test set a positive rate anywhere from roughly 0.24 to 0.29 at this size. The test set is not opened again until Task 8.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
dist = pd.DataFrame({
    "train count": y_tr.value_counts().sort_index(),
    "train share": y_tr.value_counts(normalize=True).sort_index().round(4),
    "test count":  y_te.value_counts().sort_index(),
    "test share":  y_te.value_counts(normalize=True).sort_index().round(4)})
dist.index = ["0 = No churn", "1 = Churn"]
dist.loc["total"] = [len(y_tr), 1.0, len(y_te), 1.0]
display(dist)

fig, ax = plt.subplots(figsize=(5, 3))
dist.iloc[:2][["train share", "test share"]].plot.bar(ax=ax, color=["#2E5C8A", "#BD582C"], rot=0)
ax.set_ylabel("share of split"); ax.set_title("Class distribution after stratified split"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

**Table 0b / Figure 0c.** Class counts and shares in each split. The 73/27 imbalance is moderate: not extreme enough to need resampling, but enough that accuracy and macro-F1 will rank models differently (Task 5).

### 0.4 Preprocessing (`prep`) and the CV object

`prep` is a `ColumnTransformer`: median-impute + standardise numeric columns, one-hot encode categoricals. Scaling is irrelevant for trees but matters for the logistic-regression base learner used in Tasks 4; keeping one `prep` for every model keeps the comparison fair.

`wrapped()` clones `prep` into a fresh pipeline per model so that nothing fitted in one model leaks into another.

In [ ]:
num_cols = X_tr.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_tr.select_dtypes(exclude=np.number).columns.tolist()
print(len(num_cols), "numeric:", num_cols)
print(len(cat_cols), "categorical:", cat_cols)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def wrapped(estimator):
    return Pipeline([("prep", clone(prep)), ("model", estimator)])

models = {}          # name -> unfitted model, filled as each task registers its models

def cv_report(model, name, X=None, y=None):
    '''5-fold stratified CV; registers the model under `name`; returns one row: mean±sd accuracy, macro-F1, fit time.'''
    models[name] = model
    X = X_tr if X is None else X; y = y_tr if y is None else y
    r = cross_validate(model, X, y, cv=cv, scoring=["accuracy", "f1_macro"],
                       n_jobs=-1, return_train_score=True)
    return dict(model=name,
                acc=r["test_accuracy"].mean(),   acc_sd=r["test_accuracy"].std(),
                f1=r["test_f1_macro"].mean(),    f1_sd=r["test_f1_macro"].std(),
                train_f1=r["train_f1_macro"].mean(),
                fit_s=r["fit_time"].mean())

def fmt(df):
    '''Pretty mean ± sd view of a results frame.'''
    out = pd.DataFrame({"model": df.model,
        "accuracy": [f"{m:.4f} ± {s:.4f}" for m, s in zip(df.acc, df.acc_sd)],
        "macro-F1": [f"{m:.4f} ± {s:.4f}" for m, s in zip(df.f1, df.f1_sd)],
        "train macro-F1": df.train_f1.round(4),
        "fit time (s)": df.fit_s.round(3)})
    return out

## Task 1 — Baseline: a single decision tree

A fully grown tree (`max_depth=None`) is the reference point. Watch the gap between train and CV macro-F1: that gap is the variance bagging is supposed to remove.

In [ ]:
results = []
tree = wrapped(DecisionTreeClassifier(random_state=SEED))
results.append(cv_report(tree, "Decision Tree"))
fmt(pd.DataFrame(results))

**Table 1.** Single decision tree, 5-fold stratified CV on the training set. Train macro-F1 ≈ 1.0 against a much lower CV macro-F1 is the signature of an overfitted, high-variance learner.

## Task 2 — Bagging and Random Forest

Both use 100 trees. The *only* difference between `BaggingClassifier(DecisionTreeClassifier)` and `RandomForestClassifier` with default settings is feature subsampling at each split (`max_features="sqrt"` in RF, all features in bagging). So the delta between those two rows is a direct measurement of what feature subsampling buys.

In [ ]:
bagging = wrapped(BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100, random_state=SEED, n_jobs=-1))
rf = wrapped(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))

results.append(cv_report(bagging, "Bagging (100 trees)"))
results.append(cv_report(rf, "Random Forest (100 trees)"))
fmt(pd.DataFrame(results))

**Table 2.** Baseline tree vs bagging vs Random Forest. Train macro-F1 stays near 1.0 for all three (each tree is still fully grown), but CV macro-F1 rises and its standard deviation usually falls — variance reduction without a change in the per-tree model.

*Write here:* by how much did bagging improve on the single tree? Did feature subsampling (RF vs bagging) add anything beyond that, and is the difference larger than the fold-to-fold standard deviation?

## Task 3 — AdaBoost and XGBoost with a small grid

A 3 × 3 grid over `n_estimators` and `learning_rate`, scored by macro-F1 under the same 5-fold CV. The grid is deliberately small: the point is to see the interaction (more estimators tolerate a smaller learning rate), not to squeeze out the last 0.1%.

**Caveat on the numbers you get back.** `GridSearchCV.best_score_` is the CV score of the *winning* configuration and is therefore optimistically biased — you picked the maximum of nine noisy estimates. A fair post-tuning estimate needs nested CV (see the optional cell at the end of this task). For the comparison table we accept the small bias and note it.

In [ ]:
grid = {"model__n_estimators": [50, 100, 200],
        "model__learning_rate": [0.05, 0.1, 0.3]}

def tune(estimator, name):
    gs = GridSearchCV(wrapped(estimator), grid, cv=cv, n_jobs=-1,
                      scoring={"macro_F1": "f1_macro", "F1_pos": "f1", "accuracy": "accuracy"},
                      refit="macro_F1")          # ranked and refitted by macro-F1; the other two are recorded only
    t0 = time.time(); gs.fit(X_tr, y_tr); t = time.time() - t0
    tab = (pd.DataFrame(gs.cv_results_)
             .pivot(index="param_model__n_estimators",
                    columns="param_model__learning_rate",
                    values="mean_test_macro_F1").round(4))
    print(f"{name}: best {gs.best_params_}  CV macro-F1 = {gs.best_score_:.4f}  "
          f"(grid search took {t:.1f}s)")
    return gs, tab

def config_table(gs):
    '''All grid configurations with mean CV accuracy, positive-class F1 and macro-F1, best first.'''
    r = pd.DataFrame(gs.cv_results_)
    t = pd.DataFrame({"n_estimators": r["param_model__n_estimators"].astype(int),
                      "learning_rate": r["param_model__learning_rate"].astype(float),
                      "accuracy": r["mean_test_accuracy"], "F1 (class 1)": r["mean_test_F1_pos"],
                      "macro-F1": r["mean_test_macro_F1"], "rank by macro-F1": r["rank_test_macro_F1"]})
    return t.sort_values("rank by macro-F1").reset_index(drop=True).round(4)

gs_ada, tab_ada = tune(AdaBoostClassifier(random_state=SEED), "AdaBoost")
gs_xgb, tab_xgb = tune(XGBClassifier(random_state=SEED, eval_metric="logloss",
                                     n_jobs=1, verbosity=0), "XGBoost")

#### Accuracy vs macro-F1, worked on one fold

The grid search above scored nine AdaBoost configurations on five folds each. To see *why* it was ranked by macro-F1 and not accuracy, we take **one fold** and **two configurations** and do every calculation by hand:

- **Config W** (worst by macro-F1): `n_estimators=50, learning_rate=0.05`
- **Config B** (best by macro-F1): the winner printed above

Fold 1 of the same `StratifiedKFold` is used. Its training part (about 4,507 rows) trains the model; its validation part (about 1,127 rows) plays the role of "test data" for this example. Nothing here touches the real test set.

In [ ]:
fold_tr_idx, fold_va_idx = next(iter(cv.split(X_tr, y_tr)))     # fold 1
Xf_tr, Xf_va = X_tr.iloc[fold_tr_idx], X_tr.iloc[fold_va_idx]
yf_tr, yf_va = y_tr.iloc[fold_tr_idx], y_tr.iloc[fold_va_idx]

fold_dist = pd.DataFrame({
    "fold-train count": yf_tr.value_counts().sort_index(),
    "fold-train share": yf_tr.value_counts(normalize=True).sort_index().round(4),
    "fold-validation count": yf_va.value_counts().sort_index(),
    "fold-validation share": yf_va.value_counts(normalize=True).sort_index().round(4)})
fold_dist.index = ["0 = No churn", "1 = Churn"]
fold_dist.loc["total"] = [len(yf_tr), 1.0, len(yf_va), 1.0]
display(fold_dist)
n_va, n0_va, n1_va = len(yf_va), int((yf_va == 0).sum()), int((yf_va == 1).sum())
print(f"Validation part of fold 1: {n_va} rows = {n0_va} no-churn + {n1_va} churn")

**Table 3-fold.** Class distribution inside fold 1. Both parts keep the 73/27 ratio because the fold object is stratified. Keep the validation counts in view: they are the denominators in every calculation that follows.

In [ ]:
cfg_W = dict(n_estimators=50, learning_rate=0.05)
cfg_B = {k.replace("model__", ""): v for k, v in gs_ada.best_params_.items()}
print("Config W:", cfg_W, "   Config B:", cfg_B)

fits, cms = {}, {}
for label, cfg in [("W", cfg_W), ("B", cfg_B)]:
    m = wrapped(AdaBoostClassifier(random_state=SEED, **cfg)).fit(Xf_tr, yf_tr)
    pred = m.predict(Xf_va)
    fits[label] = pred
    cms[label] = confusion_matrix(yf_va, pred, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, label, cfg in zip(axes, ["W", "B"], [cfg_W, cfg_B]):
    ConfusionMatrixDisplay(cms[label], display_labels=["No churn (0)", "Churn (1)"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Config {label}: {cfg['n_estimators']} stumps, lr {cfg['learning_rate']}\n(validation part of fold 1)", fontsize=10)
plt.tight_layout(); plt.show()

for label in ["W", "B"]:
    tn_, fp_, fn_, tp_ = cms[label].ravel()
    print(f"Config {label}:  TN={tn_:4d}  FP={fp_:4d}  FN={fn_:4d}  TP={tp_:4d}   "
          f"(predicted churn for {fp_+tp_} of {n_va} customers)")

**Figure 3-fold.** Confusion matrices of the two configurations on the validation part of fold 1. Rows are the true class, columns the predicted class. Config W's right-hand column is (almost) empty: it predicts "No churn" for essentially everyone.

#### Step-by-step calculation

The four cells of each matrix feed every metric. The cell below prints each formula with the fold's actual integers substituted, so every number can be checked with a calculator. The steps are:

1. **Accuracy** = (TP + TN) / N — the share of all customers labelled correctly.
2. **Class 1 (churn) as positive:** precision₁ = TP/(TP+FP), recall₁ = TP/(TP+FN), F1₁ = 2·P·R/(P+R).
3. **Class 0 (no churn) as positive:** swap the roles — precision₀ = TN/(TN+FN), recall₀ = TN/(TN+FP), F1₀ from those.
4. **Macro-F1** = (F1₀ + F1₁) / 2.

Convention when a denominator is zero (no predicted positives at all): precision is undefined and scikit-learn reports 0; F1 is then 0 as well.

In [ ]:
def show_steps(label):
    tn_, fp_, fn_, tp_ = (int(v) for v in cms[label].ravel())
    N = tn_ + fp_ + fn_ + tp_
    print(f"===== Config {label} =====")
    acc = (tp_ + tn_) / N
    print(f"1. accuracy      = (TP + TN) / N = ({tp_} + {tn_}) / {N} = {acc:.4f}")

    p1 = tp_ / (tp_ + fp_) if tp_ + fp_ else 0.0
    r1 = tp_ / (tp_ + fn_)
    f1_1 = 2 * p1 * r1 / (p1 + r1) if p1 + r1 else 0.0
    print(f"2. class 1 (churn) as positive:")
    print(f"   precision_1  = TP / (TP + FP) = {tp_} / ({tp_} + {fp_}) = {p1:.4f}" + ("   <- 0/0: no churn predicted, set to 0" if tp_ + fp_ == 0 else ""))
    print(f"   recall_1     = TP / (TP + FN) = {tp_} / ({tp_} + {fn_}) = {r1:.4f}")
    print(f"   F1_1         = 2 * {p1:.4f} * {r1:.4f} / ({p1:.4f} + {r1:.4f}) = {f1_1:.4f}")

    p0 = tn_ / (tn_ + fn_)
    r0 = tn_ / (tn_ + fp_)
    f1_0 = 2 * p0 * r0 / (p0 + r0)
    print(f"3. class 0 (no churn) as positive:")
    print(f"   precision_0  = TN / (TN + FN) = {tn_} / ({tn_} + {fn_}) = {p0:.4f}")
    print(f"   recall_0     = TN / (TN + FP) = {tn_} / ({tn_} + {fp_}) = {r0:.4f}")
    print(f"   F1_0         = 2 * {p0:.4f} * {r0:.4f} / ({p0:.4f} + {r0:.4f}) = {f1_0:.4f}")

    mf1 = (f1_0 + f1_1) / 2
    print(f"4. macro-F1      = (F1_0 + F1_1) / 2 = ({f1_0:.4f} + {f1_1:.4f}) / 2 = {mf1:.4f}")

    # verify against scikit-learn
    pred = fits[label]
    assert abs(acc - accuracy_score(yf_va, pred)) < 1e-12
    assert abs(f1_1 - f1_score(yf_va, pred, pos_label=1)) < 1e-12
    assert abs(f1_0 - f1_score(yf_va, pred, pos_label=0)) < 1e-12
    assert abs(mf1 - f1_score(yf_va, pred, average="macro")) < 1e-12
    print("   (all four agree with scikit-learn)\n")
    return dict(config=label, accuracy=acc, **{"F1 class 1": f1_1, "F1 class 0": f1_0, "macro-F1": mf1})

summary = pd.DataFrame([show_steps("W"), show_steps("B")]).set_index("config").round(4)
summary.loc["difference B - W"] = summary.loc["B"] - summary.loc["W"]
display(summary)

**Table 3-steps.** The two configurations side by side, with the difference in the last row.

#### What the numbers say

**The problem with accuracy.** Config W predicts "No churn" for essentially every customer. Its accuracy is still about 0.73, because 73% of the validation customers really are non-churners and it gets every one of them right by default. Accuracy is reporting the class balance, not the model. The company that deploys Config W would retain zero customers, and its metric would say "73% correct".

**What macro-F1 does differently.** It scores the two classes separately and then averages them with equal weight:

- F1 for the churn class is 0 (or near 0) for Config W, because it finds no churners.
- F1 for the no-churn class is high for *both* configurations, because that class is easy.
- Averaging the two pulls Config W down to about 0.42 and lets Config B's real work on churners show up as about 0.72.

So the difference between the two configurations is roughly 0.07 in accuracy but roughly 0.30 in macro-F1. Accuracy sees about a quarter of the change because the change happened in the minority class, which accuracy weights by its size (27%) and macro-F1 weights by half (50%).

**Why not just F1 for class 1?** F1₁ alone would also expose Config W. But it ignores class 0 entirely, so a model that flags *every* customer as a churner gets recall₁ = 1, a non-trivial F1₁, and would never be penalised for the false alarms it generates on the majority class. Macro-F1 keeps both classes in the score, which is what you want when the ranking has to be defensible on both.

This is the reason `GridSearchCV` in this notebook ranks by `f1_macro`, and the reason Table 5 sorts by it.

#### The same pattern over all nine configurations and all five folds

**Table 3-best.** All nine AdaBoost configurations on the same folds, best macro-F1 first, with the majority-class classifier as the floor. Compare the three *spread* numbers printed underneath: the range of accuracy across the grid is a fraction of the range of macro-F1, which is the evidence that accuracy cannot see most of what the tuning changed.

In [ ]:
print("AdaBoost grid (rows n_estimators, cols learning_rate, cells mean CV macro-F1):")
display(tab_ada)
print("XGBoost grid:")
display(tab_xgb)

**Table 3a/3b.** Mean CV macro-F1 over the 3 × 3 grid for AdaBoost and XGBoost. Read along a row to see the learning-rate effect at fixed ensemble size, down a column to see the effect of adding estimators.

In [ ]:
ada = gs_ada.best_estimator_     # already refitted on the full training set
xgb = gs_xgb.best_estimator_
results.append(cv_report(ada, f"AdaBoost {gs_ada.best_params_['model__n_estimators']}@lr{gs_ada.best_params_['model__learning_rate']}"))
results.append(cv_report(xgb, f"XGBoost {gs_xgb.best_params_['model__n_estimators']}@lr{gs_xgb.best_params_['model__learning_rate']}"))
fmt(pd.DataFrame(results))

**Table 3c.** Tuned boosters added to the comparison. Read the train macro-F1 column against the CV column: AdaBoost on depth-1 stumps has a small train–CV gap (low capacity, so little variance to remove), XGBoost on deeper trees a larger one. Also look at the top-left corner of the AdaBoost grid — few stumps at a low learning rate barely move the ensemble off the majority-class prediction, which is why macro-F1 there collapses towards the all-"No" value of ≈0.42.

*Optional — nested CV for an unbiased post-tuning estimate (slow: 5 × 9 × 5 = 225 fits per model):*

In [ ]:
RUN_NESTED = False      # set True if you want the unbiased number for the report
if RUN_NESTED:
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED + 1)
    for est, name in [(AdaBoostClassifier(random_state=SEED), "AdaBoost"),
                      (XGBClassifier(random_state=SEED, eval_metric="logloss", n_jobs=1, verbosity=0), "XGBoost")]:
        inner = GridSearchCV(wrapped(est), grid, scoring="f1_macro", cv=cv, n_jobs=-1)
        r = cross_validate(inner, X_tr, y_tr, cv=outer, scoring="f1_macro")
        print(f"{name} nested-CV macro-F1 = {r['test_score'].mean():.4f} ± {r['test_score'].std():.4f}")

## Task 4 — Soft voting and stacking over heterogeneous base learners

Base learners: logistic regression, Random Forest, XGBoost (the tuned one). These are deliberately different families — a voting ensemble of three Random Forests gains almost nothing because their errors are correlated.

- **Soft voting** averages predicted probabilities. Each member gets the same weight whether or not it deserves it.
- **Stacking** trains a logistic-regression meta-learner on the *out-of-fold* probabilities of the base learners (`cv=cv` inside `StackingClassifier`). Because the meta-features are out-of-fold, the meta-learner is not trained on probabilities the base learners produced for examples they had already memorised. That is the whole reason stacking is not just "voting with learned weights" — the learned weights would be worthless if fitted on in-sample probabilities.

Both ensembles are wrapped as a whole; each base learner carries its own `prep` clone.

In [ ]:
base = [("lr",  wrapped(LogisticRegression(max_iter=1000))),
        ("rf",  wrapped(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))),
        ("xgb", clone(xgb))]

voting   = VotingClassifier(base, voting="soft", n_jobs=-1)
stacking = StackingClassifier(base, final_estimator=LogisticRegression(max_iter=1000),
                              cv=cv, stack_method="predict_proba", n_jobs=-1)

results.append(cv_report(wrapped(LogisticRegression(max_iter=1000)), "Logistic Regression"))
results.append(cv_report(voting,   "Soft Voting (LR+RF+XGB)"))
results.append(cv_report(stacking, "Stacking (LR+RF+XGB -> LR)"))
fmt(pd.DataFrame(results))

**Table 4.** Logistic regression (shown so the ensembles can be judged against their weakest member) plus the two meta-ensembles.

## Task 5 — The full comparison: mean ± sd of accuracy and macro-F1, plus fit time

The table and the error-bar plot below are the central deliverable of Tasks 1–5. Two things to read off them:

1. **Is the best model actually better?** If two models' macro-F1 intervals (mean ± sd) overlap heavily, five folds cannot separate them. Say so in the report rather than declaring a winner by 0.002.
2. **Accuracy vs macro-F1 disagree.** On a 73/27 dataset a model can gain accuracy by predicting more "No" and *lose* macro-F1. Sort by both and see whether the ranking changes.

In [ ]:
result = pd.DataFrame(results)
print("Sorted by macro-F1:")
display(fmt(result.sort_values("f1", ascending=False)).reset_index(drop=True))
print("Ranking by accuracy :", list(result.sort_values("acc", ascending=False).model))
print("Ranking by macro-F1 :", list(result.sort_values("f1",  ascending=False).model))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
order = result.sort_values("f1")
for ax, col, sd, title in [(axes[0], "acc", "acc_sd", "Accuracy (5-fold CV)"),
                           (axes[1], "f1", "f1_sd", "Macro-F1 (5-fold CV)")]:
    ax.barh(order.model, order[col], xerr=order[sd], capsize=4, color="#2E5C8A")
    ax.set_title(title); ax.set_xlim(order[col].min() - 0.05, 1.0)
    for i, (m, s) in enumerate(zip(order[col], order[sd])):
        ax.text(m + s + 0.003, i, f"{m:.3f}", va="center", fontsize=8)
axes[2].barh(order.model, order.fit_s, color="#BD582C")
axes[2].set_title("Mean fit time per fold (s)"); axes[2].set_xscale("log")
for ax in axes[1:]: ax.set_yticklabels([])
plt.tight_layout(); plt.show()

**Figure 1.** CV accuracy and macro-F1 (bars = mean, whiskers = ± 1 sd across the five folds) and mean fit time per fold on a log scale. Stacking costs roughly (number of base learners × (K + 1)) fits; whether its gain over soft voting justifies that is a question the report should answer with numbers from this figure.

## Task 6 — Confusion matrix and hand-computed metrics for the best model

The "best model" is the one with the highest mean CV macro-F1 in Table 5. It is chosen here, **before** the test set is opened, and it is not changed afterwards.

To compute the metrics by hand we need predictions that were not used for training. We use **out-of-fold predictions** on the training set (`cross_val_predict`): each training example is predicted by the one fold-model that did not see it. The confusion matrix below is therefore an honest matrix from 5,634 training examples, not a test-set matrix (that comes in Task 8) and not an in-sample matrix (which for a forest would be nearly perfect and meaningless).

Row/column convention: `confusion_matrix(y_true, y_pred, labels=[0, 1])` gives

|          | pred 0 | pred 1 |
|----------|--------|--------|
| **true 0** | TN | FP |
| **true 1** | FN | TP |

In [ ]:
best_name = result.loc[result.f1.idxmax(), "model"]
chosen = models[best_name]
print("Chosen by CV macro-F1:", best_name)

# Out-of-fold probabilities for the positive class (and hard labels at the default 0.5)
oof = cross_val_predict(chosen, X_tr, y_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
oof_pred = (oof >= 0.5).astype(int)
cm = confusion_matrix(y_tr, oof_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print("Out-of-fold confusion matrix (threshold 0.5):\n", cm)
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

In [ ]:
# ---- By hand, from the four entries ----
precision_h   = tp / (tp + fp)
recall_h      = tp / (tp + fn)            # a.k.a. sensitivity / TPR
specificity_h = tn / (tn + fp)            # TNR
f1_h          = 2 * precision_h * recall_h / (precision_h + recall_h)
accuracy_h    = (tp + tn) / (tp + tn + fp + fn)

# ---- scikit-learn ----
precision_s = precision_score(y_tr, oof_pred)
recall_s    = recall_score(y_tr, oof_pred)
f1_s        = f1_score(y_tr, oof_pred)
accuracy_s  = accuracy_score(y_tr, oof_pred)
specificity_s = recall_score(y_tr, oof_pred, pos_label=0)   # recall of class 0 == specificity

check = pd.DataFrame({"by hand": [precision_h, recall_h, specificity_h, f1_h, accuracy_h],
                      "sklearn": [precision_s, recall_s, specificity_s, f1_s, accuracy_s]},
                     index=["precision", "recall (sensitivity)", "specificity", "F1 (positive class)", "accuracy"])
check["|diff|"] = (check["by hand"] - check["sklearn"]).abs()
assert (check["|diff|"] < 1e-12).all(), "hand computation disagrees with sklearn"
display(check)

fig, ax = plt.subplots(figsize=(4.2, 3.8))
ConfusionMatrixDisplay(cm, display_labels=["No churn (0)", "Churn (1)"]).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"{best_name}\nout-of-fold, threshold 0.5"); plt.tight_layout(); plt.show()

**Figure 2 / Table 6.** Out-of-fold confusion matrix for the chosen model at the default 0.5 threshold, and the five metrics computed by hand against scikit-learn (the assert guarantees they agree to machine precision). Note that `specificity` is simply `recall_score` with `pos_label=0` — scikit-learn has no separate function for it.

*Write here:* which error dominates at 0.5, FP or FN? For churn, which is the expensive one? (That answer drives Task 8.)

## Task 7 — ROC and precision–recall curves

Both curves are drawn from the same out-of-fold scores. Two numbers are reported for the PR curve on purpose:

- **Average precision (AP)** — the step-wise area, `average_precision_score`. This is the one to quote.
- **Trapezoidal PR-AUC** — `auc(recall, precision)`. Linear interpolation between PR points is not the right interpolation for precision (precision is not linear in recall), so this number differs from AP — by little on a smooth curve, by more on a jagged one. Report it under its own name; do not call it "AP".

The dashed baselines are what a random classifier achieves: ROC-AUC 0.5, and AP equal to the positive rate.

In [ ]:
fpr, tpr, _ = roc_curve(y_tr, oof)
prec, rec, _ = precision_recall_curve(y_tr, oof)
roc_auc = roc_auc_score(y_tr, oof)
ap      = average_precision_score(y_tr, oof)
pr_auc_trap = auc(rec, prec)
print(f"ROC AUC           = {roc_auc:.4f}")
print(f"Average precision = {ap:.4f}   (random baseline = positive rate = {y_tr.mean():.4f})")
print(f"Trapezoidal PR AUC = {pr_auc_trap:.4f}   (linear interpolation; differs from AP by {pr_auc_trap-ap:+.4f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(fpr, tpr, color="#2E5C8A", label=f"{best_name} (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", label="random (AUC = 0.5)")
axes[0].set(xlabel="False positive rate (1 - specificity)", ylabel="True positive rate (recall)", title="ROC curve (out-of-fold)")
axes[0].legend(loc="lower right")
axes[1].step(rec, prec, where="post", color="#BD582C", label=f"{best_name} (AP = {ap:.3f})")
axes[1].axhline(y_tr.mean(), ls="--", color="grey", label=f"random (AP = {y_tr.mean():.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (out-of-fold)", ylim=(0, 1.02))
axes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

**Figure 3.** ROC (left) and precision–recall (right) curves from out-of-fold scores. The ROC curve looks comfortable because the majority class pads the false-positive-rate axis; the PR curve shows the real difficulty of the minority class and is the one to trust under imbalance.

## Task 8 — Choose the operating threshold on out-of-fold scores, freeze it, then open the test set once

Cost model from the lab sheet: **cost = 10 × FN + FP**. A missed churner (FN) costs ten times a wasted retention offer (FP). The threshold that minimises this cost is found by sweeping every distinct out-of-fold score — not a grid of 0.1 steps, every score, so the minimum is exact.

Then, and only then: fit the chosen model on the full training set, score the test set, apply the *frozen* threshold, and report. Nothing in this cell is allowed to look at `y_te` before the final `print` block.

In [ ]:
yt = np.asarray(y_tr)
thresholds = np.r_[0.0, np.unique(oof), np.nextafter(1.0, 2.0)]   # covers "predict all 1" ... "predict all 0"
costs = np.array([10 * ((yt == 1) & (oof < t)).sum() + ((yt == 0) & (oof >= t)).sum()
                  for t in thresholds])
threshold = float(thresholds[int(np.argmin(costs))])
cost_at_05 = 10 * ((yt == 1) & (oof < 0.5)).sum() + ((yt == 0) & (oof >= 0.5)).sum()
print(f"Frozen threshold = {threshold:.4f}")
print(f"OOF cost at chosen threshold = {costs.min()}   vs   OOF cost at 0.5 = {cost_at_05}")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(thresholds, costs, color="#233343")
ax.axvline(threshold, color="#BD582C", ls="--", label=f"argmin = {threshold:.3f}")
ax.axvline(0.5, color="grey", ls=":", label="default 0.5")
ax.set(xlabel="decision threshold", ylabel="10·FN + FP (out-of-fold)", title="Cost curve on out-of-fold scores")
ax.legend(); plt.tight_layout(); plt.show()

**Figure 4.** Expected cost as a function of the threshold, computed on out-of-fold training scores. The minimum sits well below 0.5 because the cost model punishes missed positives ten times harder than false alarms, so the classifier should be eager to flag.

In [ ]:
# ---------------- The test set is opened here, once. ----------------
best = clone(chosen).fit(X_tr, y_tr)
prob = best.predict_proba(X_te)[:, 1]

def test_block(t, label):
    pred = (prob >= t).astype(int)
    c = confusion_matrix(y_te, pred, labels=[0, 1]); tn_, fp_, fn_, tp_ = c.ravel()
    return dict(threshold=label, TN=tn_, FP=fp_, FN=fn_, TP=tp_,
                precision=precision_score(y_te, pred), recall=recall_score(y_te, pred),
                specificity=tn_/(tn_+fp_), F1=f1_score(y_te, pred),
                macro_F1=f1_score(y_te, pred, average="macro"),
                accuracy=accuracy_score(y_te, pred), cost=10*fn_+fp_)

test_tab = pd.DataFrame([test_block(0.5, "0.5 (default)"),
                         test_block(threshold, f"{threshold:.4f} (frozen)")]).set_index("threshold")
display(test_tab)
print(f"Test ROC AUC           = {roc_auc_score(y_te, prob):.4f}   (threshold-free)")
print(f"Test average precision = {average_precision_score(y_te, prob):.4f}   (threshold-free)")

**Table 8.** Held-out test metrics for the chosen model at the default threshold and at the frozen cost-optimal threshold. The default row is shown *only* for contrast; it was not used to pick anything. Compare the two `cost` entries — that is the amount the threshold choice is worth under the stated cost model — and note what it bought in recall and what it paid in precision and accuracy.

*Write here:* is the test cost reduction of the same order as the OOF cost reduction? If the test improvement is much smaller than the OOF one, the threshold has overfitted the training scores.

## Task 9 — Random Forest feature importances

Impurity-based importances from `RandomForestClassifier.feature_importances_`, fitted on the full training set. Two warnings before you interpret the plot:

1. One-hot encoding splits each categorical variable into several columns, so a categorical with many levels has its importance diluted across them (and a numeric column with many unique values has an advantage). The plot therefore shows importance *per encoded column*; an aggregated view by original variable is given underneath.
2. Impurity importance measures how much a feature was *used*, not how much it *helps on unseen data*. Permutation importance on held-out data is the better tool if the question is predictive value; it is left as an extension.

In [ ]:
rf_fit = clone(rf).fit(X_tr, y_tr)
feat_names = rf_fit.named_steps["prep"].get_feature_names_out()
imp_raw = pd.Series(rf_fit.named_steps["model"].feature_importances_, index=feat_names)  # original column order
imp = imp_raw.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
imp.head(15)[::-1].plot.barh(ax=ax, color="#3C7A5A")
ax.set(title="Random Forest impurity importance — top 15 encoded columns", xlabel="importance (sums to 1 over all columns)")
plt.tight_layout(); plt.show()

# Aggregate back to the original variables
orig = pd.Series(feat_names).str.replace(r"^(num|cat)__", "", regex=True)
orig = orig.apply(lambda s: next((c for c in cat_cols if s.startswith(c + "_")), s))
assert len(orig) == len(imp_raw)
agg = imp_raw.groupby(orig.values).sum().sort_values(ascending=False)   # group the UNSORTED series, aligned with feat_names
assert abs(agg.sum() - 1) < 1e-9
print("Top 5 original variables (one-hot columns summed):")
display(agg.head(5).round(4).to_frame("importance"))

**Figure 5 / Table 9.** Top-15 encoded columns (bar chart) and the top-5 original variables after summing one-hot columns. Expect the three continuous columns (`TotalCharges`, `tenure`, `MonthlyCharges`) near the top — partly predictive value, partly the many-split-points advantage described above. Discuss in the report whether the top-5 agree with the churn intuition (month-to-month contracts, short tenure, fibre-optic internet) and whether `Contract` ranks where its predictive value suggests once its one-hot columns are summed.

*Write here:* comment on the top five.

## Analysis questions (answer in the report; keep the numbers from this notebook in view)

**Q1.** Explain why bagging often reduces variance and boosting can reduce bias. Why are these tendencies rather than guarantees for every dataset?

*Hints to argue from, not a template:* bagging averages B nearly-independent high-variance estimators — the variance of an average falls with B only to the extent that the estimators are uncorrelated, which is exactly why RF subsamples features. Boosting fits a sequence of weak learners to the residual error of the previous ones, so it can represent functions a single stump cannot; but if the weak learners are already unbiased on that data there is nothing left to reduce, and if the labels are noisy, re-weighting mistakes amplifies the noise. Use Table 2 and Table 3c as evidence.

**Q2.** A screening test reports 99.2% accuracy on a dataset with 0.8% positives. Why is that number almost meaningless, and which two metrics would you demand instead?

*Hint:* the all-negative classifier scores exactly 99.2%. Name the two metrics and say what each one would reveal about that classifier.

**Q3.** Lower the decision threshold from 0.5 to 0.2 on fixed scores. Explain why recall cannot decrease, and why precision often decreases but is not guaranteed to do so.

*Hint:* lowering the threshold can only move examples from "predicted 0" to "predicted 1". Trace what that does to TP, FN, FP and TN separately, then to each ratio. For the precision part, construct the case where every example between 0.2 and 0.5 is a true positive.

In [ ]:
# Q3 can be checked numerically on the out-of-fold scores: sweep the threshold and watch each count.
rows = []
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    p = (oof >= t).astype(int); c = confusion_matrix(yt, p, labels=[0, 1]); tn_, fp_, fn_, tp_ = c.ravel()
    rows.append(dict(threshold=t, TP=tp_, FN=fn_, FP=fp_, TN=tn_, recall=tp_/(tp_+fn_), precision=tp_/(tp_+fp_)))
q3 = pd.DataFrame(rows).set_index("threshold")
assert q3.recall.is_monotonic_increasing, "recall decreased when the threshold was lowered - impossible on fixed scores"
display(q3.round(4))

**Table Q3.** Counts and ratios as the threshold is lowered on fixed out-of-fold scores. TP and FP can only rise, FN and TN can only fall — hence recall is non-decreasing. Precision moves with the ratio of new TPs to new FPs in each band, which is why it usually but not necessarily falls.

## Deliverables checklist
- [ ] Notebook runs top to bottom with `SEED = 42` (Kernel → Restart & Run All before submitting).
- [ ] Tables 1–9 and Figures 1–5 present, each with its caption cell filled in.
- [ ] Report in the Section 3 format with written answers to Q1–Q3.
- [ ] Class mapping (which label is 1) stated in the report.
- [ ] The test set appears only in Task 8.

*Marks: Implementation 4 · Result analysis 3 · Report and discussion 3 · Total 10.*